# Diagnostic 2: img_proj_dim capacity sweep (CLIP, BASED)

In [2]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [3]:
import json

import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal import MultimodalGRUPredictor

SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
PROJ_DIMS = [4, 16, 32, 64, 128, 256]
ENCODER_NAME = 'clip'
IMG_FEAT_DIM = 768
EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '4_diagnostic' / '2_diagnostic'
RESULTS_DIR = EXPERIMENT_DIR / 'results'
SWEEP_SUMMARY_PATH = RESULTS_DIR / 'diagnostic2_capacity_sweep_clip_summary.json'

In [17]:
stats = data.load_cgm_stats()
train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)

val_participant_ids = data.load_multimodal_participant_ids('val')
VAL_PARTICIPANT_IDS = sorted(set(val_participant_ids))

assert len(val_participant_ids) == len(val_loader.dataset)

sample_batch = next(iter(train_loader))
print('batch shapes:', [tuple(x.shape) for x in sample_batch])

print('number of validation participants:', len(VAL_PARTICIPANT_IDS))
print('validation participant IDs:', VAL_PARTICIPANT_IDS)
print()

print('Diagnostic 2 uses train + validation only; the held-out test set is not loaded.')

batch shapes: [(256, 24), (256,), (256,), (256, 768), (256, 5)]
number of validation participants: 6
validation participant IDs: [np.str_('CGMacros-005'), np.str_('CGMacros-016'), np.str_('CGMacros-027'), np.str_('CGMacros-044'), np.str_('CGMacros-046'), np.str_('CGMacros-048')]

Diagnostic 2 uses train + validation only; the held-out test set is not loaded.


In [18]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    return (model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device)),
            target.to(device))

In [19]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for proj_dim in PROJ_DIMS:
    print(f"\n{'=' * 20} img_proj_dim={proj_dim} {'=' * 20}")
    for seed in SEEDS:
        print(f"\n{'-' * 16}seed={seed}{'-' * 16}")
        train.set_seed(seed)
        model = MultimodalGRUPredictor(img_feat_dim=IMG_FEAT_DIM, img_proj_dim=proj_dim,
                                        gru_hidden_size=256, gru_num_layers=3, gru_dropout=0.2)

        model, history = train.train_model(
            model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
            lr=0.0001, weight_decay=1e-05, forward_fn=forward_fn,
            checkpoint_dir=EXPERIMENT_DIR / 'checkpoints'
                            / f'diagnostic2_proj{proj_dim}_seed{seed}',
            verbose=False, progress_bar=True, progress_desc=f'proj={proj_dim} seed={seed}',
        )
        n_epochs = len(history['train_loss'])
        val_loss_hist = history['val_loss']
        best_epoch = int(np.argmin(val_loss_hist) + 1)

        y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
        val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
        val_results_per_participant = metrics.evaluate_per_participant(
            y_pred_val, y_true_val, val_participant_ids,
            stats['cgm_mean'], stats['cgm_std'])

        seed_result = {
            'img_proj_dim': proj_dim, 'seed': seed, 'n_epochs_trained': n_epochs,
            'best_epoch': best_epoch,
            'train_loss_history': [float(x) for x in history['train_loss']],
            'val_loss_history': [float(x) for x in val_loss_hist],
            'val_results': val_results,
            'val_results_per_participant': val_results_per_participant,
        }
        all_results.append(seed_result)

        with open(RESULTS_DIR / f'diagnostic2_proj{proj_dim}_seed{seed}_results.json', 'w') as f:
            json.dump(seed_result, f, indent=2)

        val_rmse = [round(val_results[h]['rmse'], 2) for h in HORIZONS]
        print(f"proj={proj_dim:>4}  seed={seed:>3}  stopped at epoch {n_epochs:>3}/50  "
              f"best epoch={best_epoch:>3}  validation RMSE @ {HORIZONS} min = {val_rmse}")


==================== img_proj_dim=4 ====================

----------------seed=7----------------
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,030,445


proj=4 seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

proj=   4  seed=  7  stopped at epoch  33/50  best epoch= 23  validation RMSE @ [15, 30, 60, 90, 120] min = [10.85, 17.05, 24.62, 29.2, 31.85]

----------------seed=14----------------
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,030,445


proj=4 seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

proj=   4  seed= 14  stopped at epoch  19/50  best epoch=  9  validation RMSE @ [15, 30, 60, 90, 120] min = [10.69, 16.79, 24.27, 28.84, 31.67]

----------------seed=21----------------
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,030,445


...

proj= 256  seed= 56  stopped at epoch  12/50  best epoch=  2  validation RMSE @ [15, 30, 60, 90, 120] min = [11.66, 17.93, 26.04, 30.77, 33.47]

----------------seed=63----------------
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


proj=256 seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

proj= 256  seed= 63  stopped at epoch  11/50  best epoch=  1  validation RMSE @ [15, 30, 60, 90, 120] min = [12.57, 18.23, 25.72, 30.5, 33.35]

----------------seed=70----------------
device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


proj=256 seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

proj= 256  seed= 70  stopped at epoch  11/50  best epoch=  1  validation RMSE @ [15, 30, 60, 90, 120] min = [12.38, 18.14, 25.78, 30.66, 33.41]


In [20]:
sweep_summary = {}

for proj_dim in PROJ_DIMS:
    rows = [r for r in all_results if r['img_proj_dim'] == proj_dim]
    best_epochs = [r['best_epoch'] for r in rows]
    stopping_epochs = [r['n_epochs_trained'] for r in rows]

    per_horizon_rmse = {}
    for h in HORIZONS:
        # sd1: variability of the overall validation RMSE across the 10 seeds
        seed_values = [r['val_results'][h]['rmse'] for r in rows]
        mean = float(np.mean(seed_values))
        sd1 = float(np.std(seed_values, ddof=1))

        # sd2: for each validation participant, first average that participant's
        # RMSE across seeds; then take the SD across validation participants.
        participant_means = []
        for pid in VAL_PARTICIPANT_IDS:
            values_across_seeds = [
                r['val_results_per_participant'][pid][h]['rmse']
                for r in rows
            ]
            participant_means.append(float(np.mean(values_across_seeds)))
        sd2 = float(np.std(participant_means, ddof=1))

        per_horizon_rmse[h] = {
            'mean': mean,
            'sd1_across_seeds': sd1,
            'sd2_across_participants': sd2,
        }

    sweep_summary[proj_dim] = {
        'best_epoch': {
            'mean': float(np.mean(best_epochs)),
            'sd': float(np.std(best_epochs, ddof=1)),
        },
        'stopping_epoch': {
            'mean': float(np.mean(stopping_epochs)),
            'sd': float(np.std(stopping_epochs, ddof=1)),
        },
        'val_rmse': per_horizon_rmse,
    }

print('=== img_proj_dim capacity sweep (CLIP, real images, train+val only) ===')
print('RMSE is reported as mean +/- sd1 (sd2), where sd1 is across seeds and sd2 is across validation participants.')
for proj_dim in PROJ_DIMS:
    s = sweep_summary[proj_dim]
    print(f'\nProjection dim = {proj_dim}')
    print(f"best epoch = {s['best_epoch']['mean']:.2f} +/- {s['best_epoch']['sd']:.2f}")
    for h in HORIZONS:
        r = s['val_rmse'][h]
        print(
            f"{h:>3}-min: RMSE = {r['mean']:.2f} +/- "
            f"{r['sd1_across_seeds']:.2f} "
            f"({r['sd2_across_participants']:.2f}) mg/dL"
        )

with open(SWEEP_SUMMARY_PATH, 'w') as f:
    json.dump({
        'encoder': ENCODER_NAME,
        'proj_dims': PROJ_DIMS,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': VAL_PARTICIPANT_IDS,
        'rmse_reporting': 'mean +/- sd1_across_seeds (sd2_across_participants)',
        'sweep_summary': sweep_summary,
    }, f, indent=2)

print(f'\nsaved per-(proj_dim, seed) results to {RESULTS_DIR}')
print(f'saved sweep summary to {SWEEP_SUMMARY_PATH}')


=== img_proj_dim capacity sweep (CLIP, real images, train+val only) ===
RMSE is reported as mean +/- sd1 (sd2), where sd1 is across seeds and sd2 is across validation participants.

Projection dim = 4
best epoch = 22.00 +/- 6.43
 15-min: RMSE = 10.70 +/- 0.11 (2.14) mg/dL
 30-min: RMSE = 16.85 +/- 0.10 (3.67) mg/dL
 60-min: RMSE = 24.33 +/- 0.11 (7.17) mg/dL
 90-min: RMSE = 28.79 +/- 0.17 (10.43) mg/dL
120-min: RMSE = 31.53 +/- 0.16 (12.32) mg/dL

Projection dim = 16
best epoch = 11.20 +/- 3.49
 15-min: RMSE = 10.85 +/- 0.09 (2.18) mg/dL
 30-min: RMSE = 17.10 +/- 0.08 (3.69) mg/dL
 60-min: RMSE = 24.85 +/- 0.10 (7.23) mg/dL
 90-min: RMSE = 29.39 +/- 0.12 (10.47) mg/dL
120-min: RMSE = 32.01 +/- 0.14 (12.30) mg/dL

Projection dim = 32
best epoch = 10.40 +/- 2.84
 15-min: RMSE = 10.93 +/- 0.11 (2.20) mg/dL
 30-min: RMSE = 17.27 +/- 0.09 (3.78) mg/dL
 60-min: RMSE = 25.19 +/- 0.06 (7.37) mg/dL
 90-min: RMSE = 29.76 +/- 0.13 (10.58) mg/dL
120-min: RMSE = 32.30 +/- 0.16 (12.40) mg/dL

Projec

In [21]:
BASE_DIR = PROJECT_ROOT / 'experiments' / '3_initial_multimodal' / 'multimodal' / 'results' / 'gated_cgm_time'
BASE_STUB = 'gated_cgm_time'

base_seed_results = []
for seed in SEEDS:
    with open(BASE_DIR / f'{BASE_STUB}_seed{seed}_results.json') as f:
        base_seed_results.append(json.load(f))

base_summary = {}
for h in HORIZONS:
    seed_values = [r['val_results'][str(h)]['rmse'] for r in base_seed_results]
    mean = float(np.mean(seed_values))
    sd1 = float(np.std(seed_values, ddof=1))

    participant_means = []
    for pid in VAL_PARTICIPANT_IDS:
        values_across_seeds = [
            r['val_results_per_participant'][pid][str(h)]['rmse']
            for r in base_seed_results
        ]
        participant_means.append(float(np.mean(values_across_seeds)))
    sd2 = float(np.std(participant_means, ddof=1))

    base_summary[h] = {
        'mean': mean,
        'sd1_across_seeds': sd1,
        'sd2_across_participants': sd2,
    }

print('=== Delta vs. CGM+Time on validation RMSE ===')
print('CGM+Time is reported as mean +/- sd1 (sd2). Deltas are projection mean minus CGM+Time mean.')
for h in HORIZONS:
    b = base_summary[h]
    print(
        f"{h:>3}-min CGM+Time: {b['mean']:.2f} +/- "
        f"{b['sd1_across_seeds']:.2f} ({b['sd2_across_participants']:.2f}) mg/dL"
    )

print()
print(f"{'proj_dim':>10}" + ''.join(f'{h:>10}min' for h in HORIZONS))
for proj_dim in PROJ_DIMS:
    s = sweep_summary[proj_dim]
    deltas = ''.join(
        f"{s['val_rmse'][h]['mean'] - base_summary[h]['mean']:>+13.2f}"
        for h in HORIZONS
    )
    print(f"{proj_dim:>10}{deltas}")

# Save the baseline context and deltas alongside the sweep summary.
delta_summary = {
    proj_dim: {
        h: float(sweep_summary[proj_dim]['val_rmse'][h]['mean'] - base_summary[h]['mean'])
        for h in HORIZONS
    }
    for proj_dim in PROJ_DIMS
}

DELTA_SUMMARY_PATH = RESULTS_DIR / 'diagnostic2_capacity_sweep_clip_delta_vs_cgm_time.json'
with open(DELTA_SUMMARY_PATH, 'w') as f:
    json.dump({
        'baseline': 'CGM+Time',
        'baseline_summary': base_summary,
        'delta_definition': 'projection mean validation RMSE - CGM+Time mean validation RMSE',
        'delta_by_projection_dimension': delta_summary,
    }, f, indent=2)

print(f'\nsaved delta summary to {DELTA_SUMMARY_PATH}')


=== Delta vs. CGM+Time on validation RMSE ===
CGM+Time is reported as mean +/- sd1 (sd2). Deltas are projection mean minus CGM+Time mean.
 15-min CGM+Time: 10.58 +/- 0.07 (2.09) mg/dL
 30-min CGM+Time: 16.68 +/- 0.07 (3.61) mg/dL
 60-min CGM+Time: 24.07 +/- 0.06 (7.04) mg/dL
 90-min CGM+Time: 28.40 +/- 0.05 (10.27) mg/dL
120-min CGM+Time: 31.25 +/- 0.06 (12.14) mg/dL

  proj_dim        15min        30min        60min        90min       120min
         4        +0.13        +0.17        +0.26        +0.39        +0.29
        16        +0.27        +0.41        +0.78        +0.99        +0.76
        32        +0.36        +0.59        +1.12        +1.36        +1.05
        64        +0.40        +0.75        +1.38        +1.60        +1.27
       128        +0.45        +0.87        +1.68        +1.93        +1.52
       256        +1.65        +1.42        +1.68        +2.16        +2.09

saved delta summary to /content/drive/MyDrive/dissertation/experiments/4_diagnostic/2_diagnostic